# Forecast diagnostic: walkthrough

This notebook runs the toolkit on the synthetic sample data and shows each step of the pipeline:

**clean → validate → reconcile → diagnose → act → summarise**

Everything comes from `forecast_diagnostic.py`; the settings come from `config.json`. To run it on your own
export, point `config.json` at your two CSVs and run this notebook again.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import forecast_diagnostic as fd

res = fd.run(ROOT / "config.json", use_ai=False, export_outputs=False)
F = res.facts
print(f"{res.cfg['region_name']}  |  snapshot {F['snapshot_date']}  |  {F['days_left_in_quarter']} days to quarter end")
print(f"{len(res.opp_raw)} rows in, {len(res.opp)} after cleaning  |  FX rates inferred: {res.fx}")

APAC  |  snapshot 2026-10-17  |  14 days to quarter end
469 rows in, 464 after cleaning  |  FX rates inferred: {'AUD': 0.66, 'INR': 0.012, 'JPY': 0.0068, 'KRW': 0.0007, 'NZD': 0.6, 'SGD': 0.74}


## 1. Clean and validate

Every check that found something, with the rows and dollars it touched. Fixes the data can answer itself
(duplicates, currency, country codes, deal age) are applied and logged; anything that needs a human decision
is flagged instead of guessed.

In [2]:
res.dq_log[["check_id", "check", "severity", "action", "rows_affected", "usd_affected"]]

,check_id,check,severity,action,rows_affected,usd_affected
0,DQ01,Exact duplicate opportunity rows,High,Removed,5,170770.0
1,DQ03,amount_usd not converted at the currency's rate,High,Corrected (recomputed),7,-107306.0
2,DQ09,Quota assigned to vacant seats (Open Req),High,Flagged,2,640000.0
3,DQ10,Open deals owned by a vacant seat,High,Flagged - reassign,6,240591.0
4,DQ17,Open deal with a close date already in the past,High,Flagged - confirm or push close date,161,6139437.0
5,DQ05,Country spelled out instead of roster ISO code,Medium,Corrected (mapped),31,1168230.0
6,DQ12,Forecast category blank,Medium,Flagged,22,738852.0
7,DQ13,'Commit' on an early-stage deal,Medium,Flagged - not used in forecast,6,299064.0
8,DQ14,Late-stage deal marked Omitted / blank,Medium,Flagged - not used in forecast,36,1359491.0
9,DQ15,age_days inconsistent with dates (incl. negati...,Medium,Corrected (recomputed from created_date),93,3420457.0


## 2. Which roll-up reproduces the reported number?

The self-reported `forecast_category` and the stage-based calculation give very different answers. Only one
of them reconciles with the number leadership is quoting, and only after cleaning.

In [3]:
res.reconciliation[["method", "raw_pct_of_quota", "clean_pct_of_quota", "matches_reported"]]

,method,raw_pct_of_quota,clean_pct_of_quota,matches_reported
0,Closed Won only,0.402074,0.390610,False
1,Won + open 'Commit' (category),0.724654,0.707701,False
2,Won + open 'Commit' + 'Best Case' (category),0.988477,0.966231,False
3,Won + open pipeline x win probability (stage-b...,0.910802,0.890347,True


Is the category worth trusting? If it meant anything, `Commit` deals would carry a much higher win
probability than `Pipeline` deals.

In [4]:
res.category_audit[["deals", "open_usd", "avg_win_prob", "share_in_early_stage", "share_past_due"]].round(2)

,deals,open_usd,avg_win_prob,share_in_early_stage,share_past_due
forecast_category_clean,,,,,
Commit,68,2920408.0,0.51,0.09,0.57
Best Case,69,2381065.0,0.45,0.39,0.59
Pipeline,86,3452706.0,0.39,0.52,0.57
Omitted,26,1073261.0,0.46,0.31,0.73
(blank),22,738852.0,0.42,0.18,0.59


## 3. The forecast, and how fragile it is

Forecast = Closed Won + every open deal × its win probability. The scenarios show what happens if the deals
that already missed their close date are worth less than the pipeline claims.

In [5]:
res.scenarios[["scenario", "forecast_usd", "pct_of_quota", "gap_usd"]]

,scenario,forecast_usd,pct_of_quota,gap_usd
0,Stage-weighted forecast (the reconciled number),8200098.290,0.890347,-1009901.710
1,Risk-adjusted: past-due deals at 50% of stated...,6810719.925,0.739492,-2399280.075
2,Downside: past-due deals slip out entirely,5421341.560,0.588636,-3788658.440
3,Rep call: Won + 'Commit' category,6517925.000,0.707701,-2692075.000


## 4. Where the gap is

Quota comes from the roster, the forecast from the deals. Vacant seats are split out from active reps,
because a coverage gap and a performance gap need different fixes.

In [6]:
cols = ["country", "segment", "quota_usd", "vacant_quota_usd", "forecast_usd", "attainment", "gap_usd",
        "share_of_gap", "coverage_of_remaining", "past_due_share_of_weighted", "win_rate_usd"]
res.by_manager[cols].round(3)

,country,segment,quota_usd,vacant_quota_usd,forecast_usd,attainment,gap_usd,share_of_gap,coverage_of_remaining,past_due_share_of_weighted,win_rate_usd
manager_name,,,,,,,,,,,
Daniel Okafor,IN,MME,1600000,640000,1155343.95,0.722,-444656.05,0.440,1.314,0.643,0.889
Mei Lin,JP,ENT,2240000,0,1973819.81,0.881,-266180.19,0.264,1.652,0.471,0.741
Priya Raman,AU,ENT,2400000,0,2269734.18,0.946,-130265.82,0.129,2.280,0.684,0.882
Aisha Khan,KR,Acquisition,1020000,0,947274.11,0.929,-72725.89,0.072,1.916,0.984,0.807
Lucas Ferreira,NZ,SMB,950000,0,897660.16,0.945,-52339.84,0.052,2.166,0.341,0.782
Tom Baker,SG,SMB,1000000,0,956266.08,0.956,-43733.92,0.043,2.225,0.535,0.805


In [7]:
res.bridge   # quota -> forecast, one row per team, vacant seats on their own line

,label,manager_name,seat,quota_usd,forecast_usd,gap_usd
2,Daniel Okafor (IN MME) - vacant seats,Daniel Okafor,vacant seats,640000,223011.74,-416988.26
4,Mei Lin (JP ENT),Mei Lin,active reps,2240000,1973819.81,-266180.19
5,Priya Raman (AU ENT),Priya Raman,active reps,2400000,2269734.18,-130265.82
0,Aisha Khan (KR Acquisition),Aisha Khan,active reps,1020000,947274.11,-72725.89
3,Lucas Ferreira (NZ SMB),Lucas Ferreira,active reps,950000,897660.16,-52339.84
6,Tom Baker (SG SMB),Tom Baker,active reps,1000000,956266.08,-43733.92
1,Daniel Okafor (IN MME) - active reps,Daniel Okafor,active reps,960000,932332.21,-27667.79


## 5. Pipeline health: stage, cohort and hygiene

In [8]:
res.health["by_stage"]

,deals,open_usd,weighted_usd,avg_win_prob,past_due_deals,past_due_weighted_usd,median_age_days
pipeline_stage,,,,,,,
Prospecting,21,923898.0,158182.97,0.168095,15,124505.29,131.0
Qualification,69,2723826.0,797975.58,0.304638,36,401885.53,82.0
Proposal,105,3991787.0,1745260.16,0.436381,59,945281.54,90.0
Negotiation,76,2926781.0,1901162.58,0.666974,51,1307084.37,123.0


In [9]:
res.health["by_cohort"]   # deals created before the quarter vs during it

,deals,open_usd,weighted_usd,past_due_deals
cohort,,,,
Created before quarter,161,6139437.0,2778756.73,161
Created in quarter,110,4426855.0,1823824.56,0


## 6. Lists a manager can act on this week

In [10]:
show = ["opp_id", "account_name", "rep_name", "manager_name", "pipeline_stage", "close_date",
        "days_past_due", "amount_usd", "win_probability", "weighted_open_usd"]
res.actions["past_due"][show].head(10)

,opp_id,account_name,rep_name,manager_name,pipeline_stage,close_date,days_past_due,amount_usd,win_probability,weighted_open_usd
138,OPP-200266,Solstice Energy,Yuki Tanaka,Mei Lin,Negotiation,2026-08-23,55,113798.0,0.73,83072.54
248,OPP-200166,Halcyon Networks,Hana Oyelaran,Priya Raman,Negotiation,2026-08-09,69,101046.0,0.75,75784.50
60,OPP-200111,Halcyon Foods,Noel Adeyemi,Priya Raman,Proposal,2026-08-19,59,108834.0,0.56,60947.04
9,OPP-200130,Sable Partners,Ivy Chen,Priya Raman,Proposal,2026-08-22,56,96834.0,0.61,59068.74
66,OPP-200127,Tidewater Labs,Ivy Chen,Priya Raman,Proposal,2026-09-12,35,112638.0,0.50,56319.00
348,OPP-200279,Copperline Networks,Diego Moreno,Mei Lin,Negotiation,2026-09-29,18,91875.0,0.60,55125.00
166,OPP-200277,Crestline Labs,Diego Moreno,Mei Lin,Negotiation,2026-09-03,44,91094.0,0.58,52834.52
345,OPP-200133,Verdant Media,Ivy Chen,Priya Raman,Negotiation,2026-09-24,23,76147.0,0.69,52541.43
440,OPP-200115,Redwood Health,Noel Adeyemi,Priya Raman,Negotiation,2026-09-21,26,104215.0,0.49,51065.35
183,OPP-200146,Halcyon Health,Rafael Dias,Priya Raman,Negotiation,2026-08-03,75,70725.0,0.66,46678.50


In [11]:
res.actions["orphaned_deals"][show]     # open deals owned by a seat nobody sits in

,opp_id,account_name,rep_name,manager_name,pipeline_stage,close_date,days_past_due,amount_usd,win_probability,weighted_open_usd
466,OPP-200221,Meridian Labs,OPEN REQ (Nikhil Rao backfill pending),Daniel Okafor,Negotiation,2026-08-01,77,47681.0,0.88,41959.28
279,OPP-200226,Kestrel Energy,OPEN REQ (Grace Lim backfill pending),Daniel Okafor,Negotiation,2026-10-24,0,62981.0,0.48,30230.88
180,OPP-200223,Marlow Logistics,OPEN REQ (Nikhil Rao backfill pending),Daniel Okafor,Negotiation,2026-08-05,73,25826.0,0.59,15237.34
83,OPP-200227,Foundry Networks,OPEN REQ (Grace Lim backfill pending),Daniel Okafor,Proposal,2026-10-20,0,39901.0,0.38,15162.38
155,OPP-200228,Bluepeak Foods,OPEN REQ (Grace Lim backfill pending),Daniel Okafor,Proposal,2026-10-22,0,27758.0,0.29,8049.82
119,OPP-200222,Sable Media,OPEN REQ (Nikhil Rao backfill pending),Daniel Okafor,Qualification,2026-10-29,0,36444.0,0.16,5831.04


## 7. The summary

`template_summary()` writes the briefing from the facts pack with no AI involved. With an API key set,
`fd.draft_summary(F, res.cfg, use_ai=True)` asks Claude to write it instead, then checks every number it
wrote against the same facts pack and falls back to this template if anything cannot be traced.

In [12]:
from IPython.display import Markdown

Markdown(fd.template_summary(F))

## APAC forecast summary - snapshot 2026-10-17 (14 days to quarter end)

**Headline.** APAC is forecasting $8.20M against $9.21M quota (89.0%), a gap of $1.01M. 70% of the gap sits in two teams: Daniel Okafor (IN MME) at 72% and Mei Lin (JP ENT) at 88%.

**What is driving the gap**
- **Empty seats:** 2 vacant seats carry $640K of quota with only $223K forecast against it (41% of the gap). 6 open deals ($241K) have no active owner.
- **Stale pipeline:** 161 of 271 open deals are past their close date, holding $2.78M (60%) of weighted pipeline. Worst: Aisha Khan (KR Acquisition), 98% past due.
- **Losses:** Mei Lin (JP ENT) lost the most this quarter ($304K).
- **Forecast category not reliable:** 6 'Commit' deals are still early stage and 36 late-stage deals are Omitted or blank, so the call is built from stage and win probability instead.

**Risk.** If past-due deals close at half their stated odds, the region lands at 74% ($6.81M). Call the quarter as a range: 74% to 89%.

**Recommended actions**
- **Next 14 days:** reassign the 6 deals with no owner ($241K) today; review the 51 past-due Negotiation deals ($1.31M weighted): confirm a dated next step or move them out of the quarter; call the quarter as a range, 74% to 89%, not on the Commit category.
- **By day 30:** backfill the vacant seats and cover their territories in the meantime; no open deal may carry a past close date (weekly automatic flag); set entry criteria for 'Commit' (start with the 6 early-stage Commit deals).
- **By day 60:** loss review with Mei Lin's team ($304K lost this quarter); coaching plans for reps below 90% in both prior quarters (Yuki Tanaka, Clara Bianchi).

**Data confidence.** 14 data checks fired (5 high severity). 5 duplicate rows removed, 7 currency conversions corrected, 31 country codes fixed before any number was calculated.

In [13]:
check = fd.verify_numbers(fd.template_summary(F), F)
print(f"{int(check['verified'].sum())}/{len(check)} numbers in the summary trace back to the facts pack")

35/35 numbers in the summary trace back to the facts pack


## 8. Run it on your own data

1. Export your opportunities and rep roster as CSV, with the columns listed in the README.
2. Point `opportunity_file` and `roster_file` in `config.json` at them (stage names, thresholds, FX and
   country spellings live there too).
3. Run this notebook again, or `python forecast_diagnostic.py`, which also writes the Excel workbook,
   the charts and the summary into `outputs/`.